# MoMo Fraud Detection — Model Building
**Step 4 of 6**

## 0. Install Dependencies

In [ ]:
import subprocess,sys
for pkg in ['pandas','numpy','scikit-learn','xgboost','imbalanced-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('Ready')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pickle
SEED = 42

## 2. Load Feature Data

In [ ]:
df = pd.read_csv('Pay-sim_features.csv')
print('Shape:',df.shape)
print('Fraud rate:',df['isFraud'].mean().round(4))

## 3. Define Features and Target

In [ ]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']
print('Features:',X.shape[1])
print('Target distribution:')
print(y.value_counts())

## 4. Train / Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print('Train size:',X_train.shape)
print('Test size :',X_test.shape)

## 5. Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('After SMOTE:')
print(pd.Series(y_train_sm).value_counts())

## 6. Scale Features

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)
print('Scaling done')

## 7. Train Model 1 — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train_sm)
print('Logistic Regression trained')

## 8. Train Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
print('Random Forest trained')

## 9. Train Model 3 — XGBoost

In [ ]:
xgb = XGBClassifier(n_estimators=100, random_state=SEED,
                     use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_sm, y_train_sm)
print('XGBoost trained')

## 10. Save Models

In [ ]:
import pickle
with open('models/logistic_regression.pkl','wb') as f: pickle.dump(lr,f)
with open('models/random_forest.pkl','wb') as f: pickle.dump(rf,f)
with open('models/xgboost.pkl','wb') as f: pickle.dump(xgb,f)
with open('models/scaler.pkl','wb') as f: pickle.dump(scaler,f)
print('All models saved to /models/')